# Gaussian-Process Bayesian Optimization

A concise, reproducible version of the graduate course experiment: a 50-term spectral GP surrogate, SVD-based posterior draws, and Thompson-sampling minimization on [-2, 2]. The objective, noise model, analytic gradients, starts, bounds, and 30-step policy follow the original project.

The unmodified [original notebook](original_project.ipynb) retains the historical outputs. This demo introduces seed 100 and includes documented compatibility and candidate-index corrections; its trajectory is not a replay of the original unseeded run. See [implementation notes](../docs/implementation_notes.md).


## Setup

Install the pinned requirements from the repository root. This notebook uses ordinary Python cells and can also run headlessly with `python scripts/run_notebook.py`.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt

# Support launching from either the repository root or notebooks/.
repository_root = Path.cwd().resolve()
if not (repository_root / "src/gp_model.py").is_file():
    repository_root = repository_root.parent
if not (repository_root / "src/gp_model.py").is_file():
    raise RuntimeError("Open the demo from the repository root or notebooks/ directory")
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from src.gp_model import SpectralGP
from src.optimization import plot_result, run_bayesian_optimization, synthetic_objective
from src.posterior_sampling import condition_on_observations
from src.utils import make_rng


## Fixed experiment settings

The Gaussian reference-measure scale is 1.0, the kernel length scale is 0.4, and the observation-noise standard deviation is 0.001. These are prescribed values, not learned hyperparameters. The objective omits basis index 0 and uses fixed coefficients [-0.25, -2.5] on indices 1 and 2.


In [ ]:
model = SpectralGP(measure_scale=1.0, length_scale=0.4, n_terms=50)
noise_std = 1e-3
seed = 100
iterations = 30
grid = np.arange(-2.0, 2.0, 0.01)
print(f"Spectral terms: {model.n_terms}; additional evaluations: {iterations}; seed: {seed}")


## Sequential Thompson sampling

Start at [-2, 0, 2]. At each step, condition the coefficients using the reduced SVD, draw one posterior function, minimize that fixed draw using analytic gradients and five-start L-BFGS-B, observe the selected point, and append the data. This is a local numerical search, with no global-optimality certificate.


In [ ]:
result = run_bayesian_optimization(
    iterations=iterations, seed=seed, model=model, noise_std=noise_std
)
print(f"Observed {result.n_initial} initial points + {iterations} selected points = {result.x.size} observations")
print(f"Observation shapes: x={result.x.shape}, y={result.y.shape}")
print(f"Posterior coefficient-draw shape: {result.sampled_weights.shape}")
print(f"Successful local solver runs per iteration: {result.successful_starts}")


## Initial observations and posterior samples

The following draws condition only on the three initial observations. A separate local generator is used for this illustration, so plotting does not alter the BO history. The draws show latent functions; they do not add fresh observation noise.


In [ ]:
initial_x = result.x[:result.n_initial]
initial_y = result.y[:result.n_initial]
initial_posterior = condition_on_observations(model, initial_x, initial_y, noise_std)
illustration_weights = initial_posterior.sample(make_rng(101), n_samples=5)
figure, axes = plt.subplots(figsize=(9, 4))
axes.plot(grid, synthetic_objective(grid, model), color="black", linewidth=2, label="Objective")
for index, weights in enumerate(illustration_weights):
    axes.plot(grid, model.evaluate(grid, weights), alpha=0.6,
              label="Initial posterior draws" if index == 0 else None)
axes.scatter(initial_x, initial_y, color="#c94c4c", zorder=5, label="Initial observations")
axes.set(xlabel="x", ylabel="f(x)", title="Posterior function draws after three observations")
axes.legend(fontsize=8)
figure.tight_layout()
if plt.get_backend().lower() != "agg":
    plt.show()
plt.close(figure)


## Final acquisition decision in the new seeded run

As in the original plots, the observations shown are those available before the final update, and the selected candidate lies on the posterior draw. This generated figure is separate from the preserved historical representative result.


In [ ]:
figure = plot_result(result, model)
output_path = repository_root / "figures/seeded_demo.png"
output_path.parent.mkdir(parents=True, exist_ok=True)
figure.savefig(output_path, dpi=160)
if plt.get_backend().lower() != "agg":
    plt.show()
plt.close(figure)
print(f"Seeded illustration saved to: {output_path.relative_to(repository_root)}")


## Inspect the observations

The smallest noisy observation is a descriptive value from this run, not a benchmark or a certificate that the objective's global minimum has been found. Fixed seeds are tested within one numerical environment; floating-point trajectories can vary across platforms and library versions.


In [ ]:
best_index = int(np.argmin(result.y))
print(f"Lowest noisy observation: x={result.x[best_index]:.6f}, y={result.y[best_index]:.6f}")
print("Run numerical checks from the repository root: python -m unittest discover -s tests -v")
